# Módulo 01 · Ingesta de documentos

Se utiliza `fundamenta/loaders.py` para leer los documentos de NovaTech y convertirlos
en objetos `Document`, que combinan el texto con sus metadatos de origen.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
for path in sorted(DATA_DIR.iterdir()):
    print(f"{path.name:35} {path.stat().st_size:>6} bytes")

beneficios-2026.pdf                   4336 bytes
faq-soporte-it.txt                    2417 bytes
manual-del-empleado.md                3468 bytes
politica-de-gastos-y-viajes.md        2854 bytes
productos-y-precios.md                2703 bytes


## Texto plano

`load_document` selecciona el lector adecuado según la extensión y devuelve siempre
una **lista** de documentos.

In [3]:
from fundamenta.loaders import load_document

[faq] = load_document(DATA_DIR / "faq-soporte-it.txt")
print(faq.metadata)
print()
print(faq.text[:400])

{'source': 'faq-soporte-it.txt', 'type': 'txt'}

PREGUNTAS FRECUENTES DE SOPORTE IT - NOVATECH (empresa ficticia)
Última actualización: septiembre de 2026

¿Cómo contacto con el equipo de IT?
Abre un ticket en el portal interno it.novatech.example o escribe en el canal de Slack
#it-ayuda. El horario de atención es de lunes a viernes de 8:00 a 18:00 (hora de Madrid).
Para incidencias urgentes fuera de horario llama a la guardia de IT al 900 000 1


## Markdown

Además del texto, el título del documento se conserva en los metadatos.

In [4]:
[manual] = load_document(DATA_DIR / "manual-del-empleado.md")
print(manual.metadata)
print(manual)

{'source': 'manual-del-empleado.md', 'type': 'md', 'title': 'Manual del empleado de NovaTech'}
Document(source='manual-del-empleado.md', chars=3409, text='# Manual del empleado de NovaTech  > NovaTech es u'...)


## PDF: un documento por página

El PDF de beneficios tiene dos páginas, por lo que se obtienen dos `Document`. El
número de página en los metadatos permitirá citar "beneficios-2026.pdf, página 2".

In [5]:
pdf_pages = load_document(DATA_DIR / "beneficios-2026.pdf")
for page in pdf_pages:
    print(page.metadata, "->", len(page.text), "caracteres")

print()
print(pdf_pages[1].text[:350])

{'source': 'beneficios-2026.pdf', 'type': 'pdf', 'page': 1} -> 1097 caracteres
{'source': 'beneficios-2026.pdf', 'type': 'pdf', 'page': 2} -> 972 caracteres

4. Bienestar
NovaTech reembolsa hasta 30 € al mes de la cuota de gimnasio o de cualquier
actividad deportiva. Como alternativa, se puede elegir el acceso gratuito a la
plataforma de bienestar FitNova, con clases en línea de yoga y meditación.
5. Retribución flexible
En España, las personas empleadas pueden destinar parte de su salario bruto a
trans


## Carga de una carpeta completa

In [6]:
from fundamenta.loaders import load_directory

documents = load_directory(DATA_DIR)
print(f"{len(documents)} documentos cargados\n")
for doc in documents:
    page = doc.metadata.get("page", "")
    print(f"{doc.source:35} {doc.metadata['type']:4} {page!s:>2}  {len(doc.text):>5} caracteres")

6 documentos cargados

beneficios-2026.pdf                 pdf   1   1097 caracteres
beneficios-2026.pdf                 pdf   2    972 caracteres
faq-soporte-it.txt                  txt       2360 caracteres
manual-del-empleado.md              md        3409 caracteres
politica-de-gastos-y-viajes.md      md        2805 caracteres
productos-y-precios.md              md        2668 caracteres


## Normalización mínima

`normalize()` unifica los saltos de línea, elimina espacios sobrantes y colapsa las
líneas en blanco repetidas.

In [7]:
from fundamenta.loaders import normalize

raw = "Política de gastos   \r\n\r\n\r\n\r\nVersión 3.2  \r\n"
print(repr(raw))
print(repr(normalize(raw)))

'Política de gastos   \r\n\r\n\r\n\r\nVersión 3.2  \r\n'
'Política de gastos\n\nVersión 3.2'


## Volumen de la colección

Como referencia práctica, en español un token equivale aproximadamente a 4 caracteres.

In [8]:
total_chars = sum(len(doc.text) for doc in documents)
print(f"Total: {total_chars} caracteres ≈ {total_chars // 4} tokens")

Total: 13311 caracteres ≈ 3327 tokens


Esta colección cabría en un prompt, pero una organización real dispone de miles de
documentos y, para cada pregunta, solo unos pocos párrafos son relevantes. El módulo 02
divide los documentos en fragmentos para recuperar únicamente lo necesario.